# SI Figure S4: correlation of implicit (PCM) corrections across solvents and methods

**S4A** correlation across solvents (both nuclei), **S4B** one solvent vs another (¹H), **S4C**
correlation across methods (both nuclei). Cells show -log10(1-r), so 3 means r=0.999.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import delta22
import delta22_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
METHOD, BASIS, GEOM = "b3lyp_d3bj", "pcSseg2", "pbe0_tz"
dft = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
base = dft[(dft["sap_nmr_method"] == METHOD) & (dft["sap_basis"] == BASIS)
           & (dft["sap_geometry_type"] == GEOM)]
one = base[base["nucleus"] == "H"]

# solvent order that groups polar-aprotic -> polar-protic -> aromatic (matches the published panel)
ORDERED_SOLVENTS = ["chloroform", "tetrahydrofuran", "dichloromethane", "acetone", "acetonitrile",
                    "dimethylsulfoxide", "trifluoroethanol", "methanol", "TIP4P",
                    "benzene", "toluene", "chlorobenzene"]

## S4A: solvent-vs-solvent PCM correlation (¹H and ¹³C)

In [ ]:
for nucleus, label in [("H", "1H"), ("C", "13C")]:
    solvent_corr = delta22.correlation_matrix(base[base["nucleus"] == nucleus], "pcm", ["solute", "site"], "solvent")
    solvent_corr = solvent_corr.reindex(index=ORDERED_SOLVENTS, columns=ORDERED_SOLVENTS)
    vals = solvent_corr.values[np.triu_indices_from(solvent_corr.values, k=1)]
    print(f"solvent PCM correlation ({label}): mean r={np.nanmean(vals):.4f}  min r={np.nanmin(vals):.4f}")
    caption = ("Correlation coefficients between solvents across all 22 solutes.\n"
               f"PCM corrections computed with {METHOD} with the {BASIS} basis.\n"
               "A value of 3 means the coefficient is 0.999.")
    delta22_plots.plot_correlation_matrix(solvent_corr, f"Solvents are Highly Correlated ({nucleus})", caption,
                                          colormap="Reds", show_values=True,
                                          save_path=figure_path(f"si_figure_s04a_{label}.png"))

## S4B: PCM corrections, chloroform vs acetonitrile (1H) -- one square of S4A

In [ ]:
pair = one.pivot_table(index=["solute", "site"], columns="solvent", values="pcm")[
    ["chloroform", "acetonitrile"]].dropna()
delta22_plots.plot_pcm_scatter(pair["chloroform"], pair["acetonitrile"], "chloroform", "acetonitrile", "H",
                               save_path=figure_path("si_figure_s04b_1H.png"))

## S4C: method-vs-method correlation (chloroform, double hybrids excluded, ¹H and ¹³C)

In [ ]:
# one solvent, exclude the double hybrids whose PCM is the substituted reference value
for nucleus, label in [("H", "1H"), ("C", "13C")]:
    chcl3 = dft[(dft["sap_basis"] == BASIS) & (dft["sap_geometry_type"] == GEOM)
                & (dft["nucleus"] == nucleus) & (dft["solvent"] == "chloroform")
                & (~dft["sap_nmr_method"].isin(delta22.DOUBLE_HYBRID_METHODS))]
    method_corr = delta22.correlation_matrix(chcl3, "pcm", ["solute", "site"], "sap_nmr_method")
    method_order = sorted(method_corr.index)          # alphabetical, matches the published panel
    method_corr = method_corr.reindex(index=method_order, columns=method_order)
    mvals = method_corr.values[np.triu_indices_from(method_corr.values, k=1)]
    print(f"method PCM correlation ({label}): mean r={np.nanmean(mvals):.4f}  min r={np.nanmin(mvals):.4f}")
    caption = ("Correlation coefficients between NMR methods across all 22 solutes.\n"
               f"PCM corrections for chloroform with the {BASIS} basis.")
    delta22_plots.plot_correlation_matrix(method_corr, f"NMR Methods are Highly Correlated ({nucleus})", caption,
                                          colormap="Reds", show_values=True,
                                          save_path=figure_path(f"si_figure_s04c_{label}.png"))